# Video Preprocess

In [29]:
import cv2
import json
import numpy as np
import os
from tqdm import tqdm
import mediapipe as mp

In [ ]:
VID_DIR     = "../../datasets/ISL_Gifs"
INV_GLOSS   = "../../datasets/invGlossList.json"    # {"hi": ["vid1","vid7",...], ...}
# VID_DIR = r'D:\Work\text2sign\datasets\WLASL\videos'
# INV_GLOSS = '../../datasets/WLASL/WLASL_con2.json'
OUT_DIR     = "../../datasets/sl2t_data/data"
PRANJALSIR_DATA = "../../datasets/sl2t_data/data_pranjalSir/data"
SEQUENCE_LEN = 30
CONFIG_FEATURE_DIM  =  (33 + 21 + 21) * 3

### Make data dirs

In [37]:
#––– LOAD AND PREPARE OUTPUT DIRS –––#
with open(INV_GLOSS, 'r') as f:
    gloss_map = json.load(f)

actions = sorted(gloss_map.keys())
label_map = {act: i for i, act in enumerate(actions)}
video2label = {vid: label_map[gloss]
               for gloss, vids in gloss_map.items() for vid in vids}

In [38]:
import cv2
import numpy as np
import mediapipe as mp
import os
import json

FEATURE_DIM = CONFIG_FEATURE_DIM


mp_holistic = mp.solutions.holistic
holistic = mp_holistic.Holistic(
    static_image_mode=False,
    model_complexity=2,        
    smooth_landmarks=True,
    enable_segmentation=False,
    refine_face_landmarks=False,
    min_detection_confidence=0.6,
    min_tracking_confidence=0.6,
)

use_cuda = cv2.cuda.getCudaEnabledDeviceCount() > 0
if use_cuda:
    # Pre‑create GPU transform objects
    cuda_clahe = cv2.cuda.createCLAHE(clipLimit=2.0, tileGridSize=(8,8))
    cuda_bilateral = lambda src: cv2.cuda.bilateralFilter(src, d=5, sigmaColor=75, sigmaSpace=75)
    cuda_resizer = lambda src, sz: cv2.cuda.resize(src, sz, interpolation=cv2.INTER_CUBIC)

def preprocess_frame_gpu(img, target_size=(256,256)):
    # upload to GPU
    gpu = cv2.cuda_GpuMat()
    gpu.upload(img)

    # resize
    gpu = cuda_resizer(gpu, target_size)

    # convert BGR→LAB on GPU
    gpu_lab = cv2.cuda.cvtColor(gpu, cv2.COLOR_BGR2Lab)
    # split channels
    l_gpu, a_gpu, b_gpu = cv2.cuda.split(gpu_lab)

    # CLAHE on L channel
    l_eq_gpu = cuda_clahe.apply(l_gpu)
    lab_eq_gpu = cv2.cuda.merge([l_eq_gpu, a_gpu, b_gpu])

    # convert back LAB→BGR
    gpu_bgr = cv2.cuda.cvtColor(lab_eq_gpu, cv2.COLOR_Lab2BGR)

    # bilateral filter
    gpu_out = cuda_bilateral(gpu_bgr)

    # download to host
    return gpu_out.download()


#––– PREPROCESSING HELPERS –––#
def preprocess_frame(img, target_size=(256, 256)):
    """
    1. Upscale to a fixed resolution
    2. Apply CLAHE on the L-channel to boost local contrast
    3. Denoise with a bilateral filter
    """
    # resize (upsample or downsample)
    img_resized = cv2.resize(img, target_size, interpolation=cv2.INTER_CUBIC)

    # convert to LAB and apply CLAHE on L channel
    lab = cv2.cvtColor(img_resized, cv2.COLOR_BGR2LAB)
    l, a, b = cv2.split(lab)
    clahe = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8,8))
    l_eq = clahe.apply(l)
    lab_eq = cv2.merge((l_eq, a, b))
    img_eq = cv2.cvtColor(lab_eq, cv2.COLOR_LAB2BGR)

    # bilateral filter to smooth noise but keep edges
    img_denoised = cv2.bilateralFilter(img_eq, d=5, sigmaColor=75, sigmaSpace=75)
    return img_denoised

#––– KEYPOINT EXTRACTION –––#
def extract_kp(img):
    """Returns a FEATURE_DIM list of x,y,z landmarks (zero‑padded if missing)."""
    # preprocess to improve low‑res details
    # img_p = preprocess_frame(img)
    img_p = preprocess_frame_gpu(img) if use_cuda else preprocess_frame(img)

    img_rgb  = cv2.cvtColor(img_p, cv2.COLOR_BGR2RGB)
    res = holistic.process(img_rgb)
    kp = []

    # order: pose, left hand, right hand
    for lm_list, count in [
        (res.pose_landmarks, 33),
        (res.left_hand_landmarks, 21),
        (res.right_hand_landmarks,21)
    ]:
        if lm_list:
            for lm in lm_list.landmark:
                kp.extend([lm.x, lm.y, lm.z])
        else:
            kp.extend([0.0] * (count * 3))

    # pad/truncate to FEATURE_DIM
    # print(f"KP: {len(kp)}")
    if len(kp) < FEATURE_DIM:
        kp += [0.0] * (FEATURE_DIM - len(kp))
    else:
        kp = kp[:FEATURE_DIM]

    return kp

#––– LOAD AND PREPARE OUTPUT DIRS –––#
with open(INV_GLOSS, 'r') as f:
    gloss_map = json.load(f)

actions = sorted(gloss_map.keys())
label_map = {act: i for i, act in enumerate(actions)}
video2label = {vid: label_map[gloss]
               for gloss, vids in gloss_map.items() for vid in vids}

# ensure output dirs exist
for act in actions:
    os.makedirs(os.path.join(OUT_DIR, act), exist_ok=True)

for act in actions:
    os.makedirs(os.path.join(OUT_DIR, act), exist_ok=True)

#––– USAGE EXAMPLE –––#
# cap = cv2.VideoCapture("low_res_video.mp4")
# while cap.isOpened():
#     ret, frame = cap.read()
#     if not ret: break
#     keypoints = extract_kp(frame) # feed keypoints in model
# cap.release()


### Process Each videos

In [39]:
from datetime import datetime

log_dir = "logs"
log_file = os.path.join(log_dir, "file_check.log")
os.makedirs(log_dir, exist_ok=True)

for vid_id, label in tqdm(video2label.items(), total=len(video2label)):
    frames = []
    # print(f"vid id: {vid_id}")
    path = os.path.join(VID_DIR, vid_id + '.mp4')
    exists = os.path.isfile(path)
    ext = os.path.splitext(vid_id)[1].lower()

    timestamp = datetime.now().strftime("%Y-%m-%d %H:%M:%S")
    log_message = f"[{timestamp}] File '{path}' exists: {exists}\n"
    
    with open(log_file, "a") as f:
        f.write(log_message)

    if ext in ['.jpg', '.jpeg', '.png']:
        img = cv2.imread(path)
        if img is None:
            continue  
        frames.append(extract_kp(img))
    else:
        cap = cv2.VideoCapture(path)
        # print(f"Path: {path}")
        while True:
            ret, img = cap.read()
            # print(f"=======> ret: {ret}  .......... img: {img}")
            if not ret:
                break
            frames.append(extract_kp(img))
        cap.release()

    # pad/truncate sequence length
    # print(f"len frames: {len(frames)}")
    if len(frames) < SEQUENCE_LEN:
        frames += [[0.0]*FEATURE_DIM] * (SEQUENCE_LEN - len(frames))
    else:
        frames = frames[:SEQUENCE_LEN]

    # save to .npy
    out_dir = os.path.join(OUT_DIR, actions[label])
    idx = len(os.listdir(out_dir))
    np.save(os.path.join(out_dir, f"{idx}.npy"), np.array(frames))

100%|██████████| 100/100 [00:23<00:00,  4.22it/s]


### check npy

In [32]:
import os
import numpy as np

zero_count2 = 0
total2 = 0
for gloss in os.listdir(PRANJALSIR_DATA):
    gloss_path = os.path.join(PRANJALSIR_DATA, gloss)
    for fname in os.listdir(gloss_path):
        fname_path = os.path.join(gloss_path, fname)
        for file in os.listdir(fname_path):
            path = os.path.join(fname_path, file)
            arr = np.load(path)
            total2 += 1
            if np.all(arr == 0):
                zero_count2 += 1

print(f"Total files: {total2}")
print(f"Zero-only files: {zero_count2}")
print(f"Percent zeros: {100 * zero_count2 / total2:.2f}%")

Total files: 3600
Zero-only files: 2374
Percent zeros: 65.94%


In [40]:
import os
import numpy as np

zero_count = 0
total = 0
for gloss in tqdm(actions):
    gloss_dir = os.path.join(OUT_DIR, gloss)
    # print(" === ")
    # print(f"Gloss Dir: {gloss_dir}")
    for fname in os.listdir(gloss_dir):
        # print(f"File Name: {fname}")
        path = os.path.join(gloss_dir, fname)
        # print(f"Path: {path}")
        # print(" --- ")
        arr = np.load(path)
        total += 1
        if np.all(arr == 0):
            zero_count += 1

print(f"Total files: {total}")
print(f"Zero-only files: {zero_count}")
print(f"Percent zeros: {100 * zero_count / total:.2f}%")


100%|██████████| 100/100 [00:00<00:00, 703.94it/s]

Total files: 224
Zero-only files: 200
Percent zeros: 89.29%


# Model Arch and Training Loop

In [21]:
import tensorflow as tf
from keras.models import Sequential
from keras.layers import (
    Input, Conv1D, BatchNormalization, MaxPooling1D,
    Bidirectional, LSTM, Dropout, Dense, Layer
)
from keras.callbacks import EarlyStopping, ModelCheckpoint, ReduceLROnPlateau

class Attention(Layer):
    def build(self, input_shape):
        # input_shape: (batch, time, features)
        self.W = self.add_weight(
            name='att_weight',
            shape=(input_shape[-1], 1),
            initializer='random_normal',
            trainable=True
        )
        super().build(input_shape)

    def call(self, x):
        # x: (batch, time, features)
        # e = tanh(x · W) => (batch, time, 1)
        e = tf.math.tanh(tf.tensordot(x, self.W, axes=[[2], [0]]))
        # alpha = softmax(e) over time axis => (batch, time, 1)
        alpha = tf.nn.softmax(e, axis=1)
        # context = sum over time of (x * alpha) => (batch, features)
        context = tf.reduce_sum(x * alpha, axis=1)
        return context

    def compute_output_shape(self, input_shape):
        # returns (batch, features)
        return (input_shape[0], input_shape[2])


# --- model configuration ---
SEQUENCE_LEN = 20
FEATURE_DIM  = CONFIG_FEATURE_DIM
NUM_CLASSES  = len(actions) 

# --- build ---
model = Sequential([
    Input(shape=(SEQUENCE_LEN, FEATURE_DIM)),

    # Conv block 1
    Conv1D(64, 3, activation='relu', padding='same'),
    BatchNormalization(),
    MaxPooling1D(2),

    # Conv block 2
    Conv1D(128, 3, activation='relu', padding='same'),
    BatchNormalization(),
    MaxPooling1D(2),

    # Bidirectional LSTM stack
    Bidirectional(LSTM(64, return_sequences=True)),
    Dropout(0.3),
    Bidirectional(LSTM(64, return_sequences=True)),
    Dropout(0.3),

    # Attention aggregation
    Attention(),
    Dropout(0.3),

    # Classification head
    Dense(64, activation='relu'),
    Dropout(0.3),
    Dense(NUM_CLASSES, activation='softmax')
])

# compile
model.compile(
    optimizer='adam',
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

# callbacks
callbacks = [
    EarlyStopping(monitor='val_loss', patience=10, restore_best_weights=True),
    ModelCheckpoint("best_model_CNN1s-LSTM.h5", save_best_only=True, monitor='val_loss'),
    ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=5)
]

# history = model.fit(
#     X_train, Y_train,
#     validation_data=(X_test, Y_test),
#     epochs=100,
#     batch_size=32,
#     callbacks=callbacks
# )


### model2 test

In [8]:
import tensorflow as tf
from keras.models import Model
from keras.layers import (
    Input, Conv1D, BatchNormalization, Activation,
    Add, GlobalAveragePooling1D, Dense, Dropout
)
from keras.callbacks import EarlyStopping, ModelCheckpoint, ReduceLROnPlateau

# --- Model Configuration ---
SEQUENCE_LEN = 20
FEATURE_DIM  = 126
NUM_CLASSES  = len(actions)  # Replace with actual number of classes

# --- Residual Block ---
def residual_block(x, filters, kernel_size=3):
    # Shortcut path
    shortcut = x
    
    # Main path
    x = Conv1D(filters, kernel_size, padding='same')(x)
    x = BatchNormalization()(x)
    x = Activation('relu')(x)
    
    x = Conv1D(filters, kernel_size, padding='same')(x)
    x = BatchNormalization()(x)
    
    # Match dimensions if needed
    if shortcut.shape[-1] != filters:
        shortcut = Conv1D(filters, 1, padding='same')(shortcut)
    
    # Add shortcut to main path
    x = Add()([x, shortcut])
    x = Activation('relu')(x)
    return x

# --- Build Model ---
inputs = Input(shape=(SEQUENCE_LEN, FEATURE_DIM))

# Initial feature extraction
x = Conv1D(64, 7, padding='same')(inputs)
x = BatchNormalization()(x)
x = Activation('relu')(x)

# Residual blocks
x = residual_block(x, 64)
x = residual_block(x, 128, 2)  # Downsample
x = residual_block(x, 128)
x = residual_block(x, 256, 2)  # Downsample
x = residual_block(x, 256)

# Final processing
x = GlobalAveragePooling1D()(x)
x = Dense(128, activation='relu')(x)
x = Dropout(0.3)(x)
outputs = Dense(NUM_CLASSES, activation='softmax')(x)

model = Model(inputs, outputs)

# --- Compile Model ---
model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=0.001),
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

# --- Callbacks ---
callbacks = [
    EarlyStopping(monitor='val_loss', patience=15, restore_best_weights=True),
    ModelCheckpoint("best_model_TCN.h5", save_best_only=True, monitor='val_loss'),
    ReduceLROnPlateau(monitor='val_loss', factor=0.2, patience=5, min_lr=1e-6)
]

## Model Training

In [22]:
import numpy as np
import os
from sklearn.model_selection import train_test_split
from keras.utils import to_categorical

# configs
SEQUENCE_LEN = 20
FEATURE_DIM  = CONFIG_FEATURE_DIM
NUM_CLASSES  = len(actions)
OUT_DIR      = "../../datasets/sl2t_data/data"

def fix_array(arr):
    """
    Take an array of shape (SEQUENCE_LEN, D) and:
        - if D < FEATURE_DIM, pad zeros to the right
        - if D > FEATURE_DIM, truncate columns past FEATURE_DIM
    Returns an array of shape (SEQUENCE_LEN, FEATURE_DIM).
    """
    seq, dim = arr.shape
    if dim < FEATURE_DIM:
        # pad with zeros
        pad = np.zeros((seq, FEATURE_DIM - dim), dtype=arr.dtype)
        return np.concatenate([arr, pad], axis=1)
    elif dim > FEATURE_DIM:
        # truncate extra features
        return arr[:, :FEATURE_DIM]
    else:
        return arr

# gather & fix all samples
X, Y = [], []
for label, gloss in enumerate(actions):
    gloss_dir = os.path.join(OUT_DIR, gloss)
    for fname in os.listdir(gloss_dir):
        arr = np.load(os.path.join(gloss_dir, fname))  # shape (20, D)
        if arr.shape[0] != SEQUENCE_LEN:
            # if for some reason the time‑axis is off, skip
            continue
        arr_fixed = fix_array(arr)
        X.append(arr_fixed)
        Y.append(label)

# now X is a list of (20,126) arrays → safe to stack
X = np.stack(X, axis=0)                              # (N, 20, 126)
Y = to_categorical(Y, num_classes=NUM_CLASSES)

# stratified split
X_train, X_test, Y_train, Y_test = train_test_split(
    X, Y,
    test_size=0.30,
    random_state=42,
    shuffle=True
)

# and train
history = model.fit(
    X_train, Y_train,
    validation_data=(X_test, Y_test),
    epochs=100,
    batch_size=32,
    callbacks=callbacks
)


Epoch 1/100
15/15 ━━━━━━━━━━━━━━━━━━━━ 0s 14ms/step - accuracy: 0.0076 - loss: 4.8888  

15/15 ━━━━━━━━━━━━━━━━━━━━ 11s 106ms/step - accuracy: 0.0077 - loss: 4.8889 - val_accuracy: 0.0052 - val_loss: 4.8842 - learning_rate: 0.0010
Epoch 2/100
15/15 ━━━━━━━━━━━━━━━━━━━━ 0s 23ms/step - accuracy: 0.0203 - loss: 4.8490 - val_accuracy: 0.0103 - val_loss: 4.8877 - learning_rate: 0.0010
Epoch 3/100
15/15 ━━━━━━━━━━━━━━━━━━━━ 0s 23ms/step - accuracy: 0.0392 - loss: 4.8020 - val_accuracy: 0.0206 - val_loss: 4.8932 - learning_rate: 0.0010
Epoch 4/100
15/15 ━━━━━━━━━━━━━━━━━━━━ 0s 23ms/step - accuracy: 0.0397 - loss: 4.7114 - val_accuracy: 0.0206 - val_loss: 4.8864 - learning_rate: 0.0010
Epoch 5/100
14/15 ━━━━━━━━━━━━━━━━━━━━ 0s 17ms/step - accuracy: 0.0480 - loss: 4.5988

15/15 ━━━━━━━━━━━━━━━━━━━━ 1s 33ms/step - accuracy: 0.0472 - loss: 4.5981 - val_accuracy: 0.0103 - val_loss: 4.8565 - learning_rate: 0.0010
Epoch 6/100
15/15 ━━━━━━━━━━━━━━━━━━━━ 0s 24ms/step - accuracy: 0.0497 - loss: 4.5301 - val_accuracy: 0.0155 - val_loss: 4.8847 - learning_rate: 0.0010
Epoch 7/100
15/15 ━━━━━━━━━━━━━━━━━━━━ 0s 25ms/step - accuracy: 0.0376 - loss: 4.4842 - val_accuracy: 0.0052 - val_loss: 4.9523 - learning_rate: 0.0010
Epoch 8/100
15/15 ━━━━━━━━━━━━━━━━━━━━ 0s 29ms/step - accuracy: 0.0540 - loss: 4.3429 - val_accuracy: 0.0206 - val_loss: 4.9159 - learning_rate: 0.0010
Epoch 9/100
15/15 ━━━━━━━━━━━━━━━━━━━━ 0s 23ms/step - accuracy: 0.0815 - loss: 4.2521 - val_accuracy: 0.0309 - val_loss: 5.0009 - learning_rate: 0.0010
Epoch 10/100
15/15 ━━━━━━━━━━━━━━━━━━━━ 0s 23ms/step - accuracy: 0.0581 - loss: 4.2975 - val_accuracy: 0.0361 - val_loss: 4.9248 - learning_rate: 0.0010
Epoch 11/100
15/15 ━━━━━━━━━━━━━━━━━━━━ 0s 23ms/step - accuracy: 0.0485 - loss: 4.2588 - val_accura

### Model2 Training (Torch)

In [19]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from torch.optim.lr_scheduler import ReduceLROnPlateau
import numpy as np
import os
from sklearn.model_selection import train_test_split

# Configuration
SEQUENCE_LEN = 20
FEATURE_DIM = CONFIG_FEATURE_DIM
NUM_KEYPOINTS = 42  # 126 features / 3 coordinates
NUM_COORDS = 3
NUM_CLASSES = len(actions)  # Make sure 'actions' is defined
OUT_DIR = "../../datasets/sl2t_data/data"

# Device setup
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

# Data preparation functions
def fix_array(arr):
    seq, dim = arr.shape
    if dim < FEATURE_DIM:
        pad = np.zeros((seq, FEATURE_DIM - dim), dtype=arr.dtype)
        return np.concatenate([arr, pad], axis=1)
    elif dim > FEATURE_DIM:
        return arr[:, :FEATURE_DIM]
    return arr

class SkeletonDataset(Dataset):
    def __init__(self, X, Y):
        self.X = torch.from_numpy(X).float()
        self.Y = torch.from_numpy(Y).long()  # CrossEntropyLoss needs class indices
        
    def __len__(self):
        return len(self.X)
    
    def __getitem__(self, idx):
        return self.X[idx], self.Y[idx]

# Model definition
class SkeletonAction3DCNN(nn.Module):
    def __init__(self):
        super().__init__()
        
        self.features = nn.Sequential(
            # Reshape input to (batch, 1, seq_len, num_keypoints, num_coords)
            nn.Conv3d(1, 64, kernel_size=(3, 3, 3), padding=(1, 1, 1)),
            nn.BatchNorm3d(64),
            nn.ReLU(),
            nn.MaxPool3d((2, 2, 1)),
            
            nn.Conv3d(64, 128, kernel_size=(3, 3, 3), padding=(1, 1, 1)),
            nn.BatchNorm3d(128),
            nn.ReLU(),
            nn.MaxPool3d((2, 2, 1)),
            
            nn.Conv3d(128, 256, kernel_size=(3, 3, 3), padding=(1, 1, 1)),
            nn.BatchNorm3d(256),
            nn.ReLU(),
            nn.MaxPool3d((2, 2, 1)),
            
            nn.Conv3d(256, 512, kernel_size=(1, 1, 1)),
            nn.BatchNorm3d(512),
            nn.ReLU(),
        )
        
        self.classifier = nn.Sequential(
            nn.AdaptiveAvgPool3d(1),
            nn.Flatten(),
            nn.Linear(512, 256),
            nn.ReLU(),
            nn.Dropout(0.5),
            nn.Linear(256, NUM_CLASSES)
        )
    
    def forward(self, x):
        # Reshape input from (batch, seq_len, features) to 3D skeleton format
        x = x.view(-1, SEQUENCE_LEN, NUM_KEYPOINTS, NUM_COORDS)
        x = x.unsqueeze(1)  # Add channel dimension
        x = self.features(x)
        x = self.classifier(x)
        return x

# Data loading
X, Y = [], []
for label, gloss in enumerate(actions):
    gloss_dir = os.path.join(OUT_DIR, gloss)
    for fname in os.listdir(gloss_dir):
        arr = np.load(os.path.join(gloss_dir, fname))
        if arr.shape[0] == SEQUENCE_LEN:
            X.append(fix_array(arr))
            Y.append(label)

X = np.stack(X)  # (N, 20, 126)
Y = np.array(Y)  # Class indices (not one-hot)

# Train-test split
X_train, X_test, Y_train, Y_test = train_test_split(
    X, Y, test_size=0.3, random_state=42, shuffle=True
)

# Create datasets and dataloaders
train_dataset = SkeletonDataset(X_train, Y_train)
test_dataset = SkeletonDataset(X_test, Y_test)

train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=32)

# Initialize model, loss, optimizer
model = SkeletonAction3DCNN().to(device)
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)
scheduler = ReduceLROnPlateau(optimizer, 'min', patience=5, factor=0.5)

# Training loop
def train_model():
    best_acc = 0
    
    for epoch in range(100):
        model.train()
        train_loss, correct, total = 0, 0, 0
        
        for inputs, labels in train_loader:
            inputs, labels = inputs.to(device), labels.to(device)
            
            optimizer.zero_grad()
            outputs = model(inputs)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()
            
            train_loss += loss.item()
            _, predicted = outputs.max(1)
            total += labels.size(0)
            correct += predicted.eq(labels).sum().item()
        
        # Validation
        val_loss, val_correct, val_total = 0, 0, 0
        model.eval()
        with torch.no_grad():
            for inputs, labels in test_loader:
                inputs, labels = inputs.to(device), labels.to(device)
                outputs = model(inputs)
                loss = criterion(outputs, labels)
                
                val_loss += loss.item()
                _, predicted = outputs.max(1)
                val_total += labels.size(0)
                val_correct += predicted.eq(labels).sum().item()
        
        # Statistics
        train_loss /= len(train_loader)
        val_loss /= len(test_loader)
        train_acc = 100 * correct / total
        val_acc = 100 * val_correct / val_total
        
        print(f'Epoch {epoch+1}:')
        print(f'Train Loss: {train_loss:.4f} | Acc: {train_acc:.2f}%')
        print(f'Val Loss: {val_loss:.4f} | Acc: {val_acc:.2f}%')
        
        # Save best model
        if val_acc > best_acc:
            best_acc = val_acc
            torch.save(model.state_dict(), 'best_3dcnn_model.pth')
            print('Best model saved!')
        
        scheduler.step(val_loss)

# Start training
train_model()

# Load best model for evaluation
model.load_state_dict(torch.load('best_3dcnn_model.pth'))
model.eval()

Using device: cuda


RuntimeError: shape '[-1, 20, 42, 3]' is invalid for input of size 144000